# Fine-tuning with LoRA on synthetic data

Exercise notebook for the lesson **[Fine-tuning with LoRA on synthetic data](https://ramkumarmd.com/medical-ai/generative-ai-llms/fine-tuning-lora)** in the *Generative AI & LLMs* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Fine-tuning adapts a pre-trained model to a specific task by training it further on examples. LoRA (low-rank adaptation) does this cheaply: it freezes the original weights and trains small add-on matrices, often less than 1% of the parameters. Fine-tuning is best for teaching a consistent format, style or narrow skill; for up-to-date facts, retrieval is usually better. Fine-tuning on patient data raises privacy risks, because models can memorise training text.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### How LoRA works

How many parameters LoRA trains for one 4,096 × 4,096 weight matrix.

In [ ]:
d = 4096   # width of one weight matrix in a mid-sized language model layer
for rank in [4, 8, 16, 64]:
    full = d * d
    lora = 2 * d * rank          # A is r x d, B is d x r
    print(f"rank {rank:>2}: LoRA trains {lora:>9,} parameters vs {full:,} ({lora / full:.2%})")

# rank  4: LoRA trains    32,768 parameters vs 16,777,216 (0.20%)
# rank  8: LoRA trains    65,536 parameters vs 16,777,216 (0.39%)
# rank 16: LoRA trains   131,072 parameters vs 16,777,216 (0.78%)
# rank 64: LoRA trains   524,288 parameters vs 16,777,216 (3.12%)

### LoRA in NumPy

Train LoRA adapters of rank 1, 2, 3 and 8 on a task that needs a rank-3 change.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
d = 64
W0 = rng.normal(0, 1 / np.sqrt(d), (d, d))                 # frozen "pre-trained" weights
U, V = rng.normal(0, 0.3, (d, 3)), rng.normal(0, 0.3, (3, d))
true_change = U @ V                                          # the new task needs a rank-3 change
X = rng.normal(size=(2000, d))
Y = X @ (W0 + true_change).T

def train_lora(rank, steps=1000, lr=0.05):
    r = np.random.default_rng(1)
    A = r.normal(0, 0.01, (rank, d))   # trained
    B = np.zeros((d, rank))            # trained; starts at zero so the model starts as W0
    for _ in range(steps):
        err = X @ (W0 + B @ A).T - Y
        grad = err.T @ X / len(X)
        B, A = B - lr * grad @ A.T, A - lr * B.T @ grad
    return np.mean(err**2), A.size + B.size

print(f"No adaptation:  error {np.mean((X @ W0.T - Y)**2):.3f}")
for rank in [1, 2, 3, 8]:
    error, n_params = train_lora(rank)
    print(f"LoRA rank {rank}:    error {error:.3f}   trained parameters {n_params} of {W0.size}")

# No adaptation:  error 1.596
# LoRA rank 1:    error 0.886   trained parameters 128 of 4096
# LoRA rank 2:    error 0.351   trained parameters 256 of 4096
# LoRA rank 3:    error 0.000   trained parameters 384 of 4096
# LoRA rank 8:    error 0.000   trained parameters 1024 of 4096

## Exercise data

Synthetic data written for this exercise (no real patients). Run this cell before starting the steps.

In [ ]:
# Synthetic note-to-summary pairs for the optional fine-tuning step (no real patients).
# The first 12 are for training, the last 4 are held out for comparison.
note_summary_pairs = [
    ("58M T2DM 10y. Metformin 1 g BD. HbA1c 8.6% (was 7.9%). BP 142/88. eGFR 72. Plan: add sitagliptin, diet referral, review 3 mo.",
     "Type 2 diabetes, HbA1c rising to 8.6% on metformin. Sitagliptin added; dietitian referral; review in 3 months."),
    ("34F T1DM since 15. Basal-bolus. 3 hypos/wk, mostly night. HbA1c 6.9%. Plan: reduce basal 10%, CGM review 2 wk.",
     "Type 1 diabetes with frequent night-time hypoglycaemia. Basal insulin reduced by 10%; CGM review in 2 weeks."),
    ("66F T2DM, CKD 3a (eGFR 52), UACR 45 mg/mmol. On metformin. HbA1c 7.4%. Plan: start empagliflozin 10 mg, recheck U&E 4 wk.",
     "Type 2 diabetes with CKD stage 3a and albuminuria. Empagliflozin 10 mg started; kidney function check in 4 weeks."),
    ("29F, 26/40 pregnant. OGTT fasting 5.8, 2h 8.9 mmol/L. Plan: GDM diet advice, home glucose monitoring, review 1 wk.",
     "Gestational diabetes diagnosed at 26 weeks. Diet advice and home glucose monitoring; review in 1 week."),
    ("71M T2DM 20y. Insulin glargine 24 u. Numb feet both sides, monofilament absent L. No ulcers. Plan: podiatry, foot education.",
     "Long-standing type 2 diabetes on glargine with reduced sensation in both feet. Podiatry referral and foot-care education."),
    ("45M T2DM new dx. HbA1c 9.8%. BMI 33. No symptoms. Plan: metformin titrate to 1 g BD, structured education, review 3 mo.",
     "New type 2 diabetes, HbA1c 9.8%. Metformin started and titrated; structured education; review in 3 months."),
    ("52F T2DM. HbA1c 7.1%. LDL 3.4 mmol/L, not on statin. Plan: discussed statin, patient agreed atorvastatin 20 mg.",
     "Type 2 diabetes, HbA1c 7.1%. Not on a statin with LDL 3.4; atorvastatin 20 mg started after discussion."),
    ("38M T1DM. DKA admission last month after missed doses. HbA1c 11.2%. Plan: sick-day rules, psychology referral, review 2 wk.",
     "Type 1 diabetes, recent DKA after missed insulin, HbA1c 11.2%. Sick-day rules taught; psychology referral; review in 2 weeks."),
    ("60F T2DM. Retinal screen: background retinopathy R. HbA1c 7.8%. BP 150/90. Plan: start ramipril 2.5 mg, rescreen 12 mo.",
     "Type 2 diabetes with background retinopathy and raised blood pressure. Ramipril 2.5 mg started; repeat eye screening in 12 months."),
    ("48M T2DM. On gliclazide + metformin. Weight up 6 kg. HbA1c 8.0%. Plan: stop gliclazide, start semaglutide, review 6 wk.",
     "Type 2 diabetes with weight gain on gliclazide. Gliclazide stopped and semaglutide started; review in 6 weeks."),
    ("75F T2DM, frail, lives alone. HbA1c 6.2% on gliclazide. Two falls. Plan: stop gliclazide, relax target, review 4 wk.",
     "Frail older woman with type 2 diabetes and falls on gliclazide, HbA1c 6.2%. Gliclazide stopped; less strict target; review in 4 weeks."),
    ("41F LADA suspected. GAD antibodies positive. HbA1c 8.9%. Plan: start basal insulin 10 u, education, review 2 wk.",
     "Probable LADA with positive GAD antibodies, HbA1c 8.9%. Basal insulin 10 units started with education; review in 2 weeks."),
    # held out
    ("63M T2DM. Foot ulcer R hallux 1 cm, no infection signs. HbA1c 8.3%. Plan: offloading, dressing, podiatry within 1 wk.",
     "Type 2 diabetes with a 1 cm right big-toe ulcer without infection. Offloading, dressings and podiatry within 1 week."),
    ("27M T1DM. Starting pump. HbA1c 7.6%. Plan: pump training booked, carb counting refresher, review after training.",
     "Type 1 diabetes, HbA1c 7.6%, starting an insulin pump. Pump training and carbohydrate-counting refresher booked."),
    ("55F T2DM. eGFR 38. On metformin 1 g BD. HbA1c 7.2%. Plan: reduce metformin to 500 mg BD, recheck eGFR 3 mo.",
     "Type 2 diabetes with eGFR 38. Metformin reduced to 500 mg twice daily; kidney function recheck in 3 months."),
    ("69M T2DM. Chest tightness on exertion 2 wk. ECG today normal. Plan: urgent cardiology referral, GTN spray.",
     "Type 2 diabetes with two weeks of exertional chest tightness, ECG normal. Urgent cardiology referral; GTN spray given."),
]
train_pairs, held_out_pairs = note_summary_pairs[:12], note_summary_pairs[12:]
print(f"{len(train_pairs)} training pairs, {len(held_out_pairs)} held-out pairs")

## Exercise

**Goal:** Implement and explore LoRA in NumPy, then plan a responsible clinical fine-tuning project.

**Data:** Synthetic matrices and examples generated in the notebook (Generated in the notebook)

### Step 1

Reproduce the rank comparison and plot error against rank.

In [ ]:
# Your code here


### Step 2

Add noise to the targets and use a held-out set to see whether a large rank overfits.

In [ ]:
# Your code here


### Step 3

Apply a LoRA adapter, then 'merge' it into W0 (W0 + B @ A) and confirm the outputs are identical.

In [ ]:
# Your code here


### Step 4

Optional, with a GPU: fine-tune a small open model with a LoRA adapter (Hugging Face PEFT) on the provided synthetic note-to-summary pairs, and compare with the base model plus prompting on held-out examples.

In [ ]:
# Your code here


### Step 5

Write a one-page plan covering data source, de-identification, evaluation, safety testing and who approves deployment.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/generative-ai-llms/fine-tuning-lora](https://ramkumarmd.com/medical-ai/generative-ai-llms/fine-tuning-lora). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.